In [5]:
## 1. Datenimport und Vorbereitung

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm.auto import tqdm

tqdm.pandas()

df = pd.read_excel("Daten ohne BERT.xlsm")

df.head()

,product_id,product_name,good_type,price (USD),stars,review_text,Nr
0,1,SanDisk 128GB Ultra Flair USB 3.0 Flash Drive ...,search,19.88,5,Works great!,1
1,1,SanDisk 128GB Ultra Flair USB 3.0 Flash Drive ...,search,19.88,5,Very sleek and stylish large capacity easy to ...,2
2,1,SanDisk 128GB Ultra Flair USB 3.0 Flash Drive ...,search,19.88,5,Thin material!,3
3,1,SanDisk 128GB Ultra Flair USB 3.0 Flash Drive ...,search,19.88,4,"Worked great, just be aware that it is TINY!",4
4,1,SanDisk 128GB Ultra Flair USB 3.0 Flash Drive ...,search,19.88,1,not the right item for my computer,5


In [6]:
## Prüfung der Datenstruktur

df.info()
df["good_type"].value_counts()
df["stars"].value_counts().sort_index()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    1000 non-null   int64  
 1   product_name  1000 non-null   str    
 2   good_type     1000 non-null   str    
 3   price (USD)   1000 non-null   float64
 4   stars         1000 non-null   int64  
 5   review_text   1000 non-null   str    
 6   Nr            1000 non-null   int64  
dtypes: float64(1), int64(3), str(3)
memory usage: 54.8 KB


stars
1     55
2     12
3     43
4     80
5    810
Name: count, dtype: int64

In [7]:
## Bereinigung der Rezensionstexte

df["review_text_clean"] = df["review_text"].fillna("").astype(str).str.strip()

In [12]:
## Laden des BERT-Modells

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(105879, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [8]:
## Funktion zur Berechnung der Sentimentwerte

def get_bert_sentiment(text):
    if not isinstance(text, str) or text.strip() == "":
        return pd.Series([None, None, None, None, None, None, None])

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits.detach().numpy()[0]
    probs = softmax(logits)

    bert_discrete = int(probs.argmax() + 1)
    bert_continuous = sum((i + 1) * probs[i] for i in range(5))
    confidence = probs.max()

    return pd.Series([
        bert_discrete,
        bert_continuous,
        confidence,
        probs[0],
        probs[1],
        probs[2],
        probs[3],
        probs[4]
    ])

In [13]:
### Durchführung der Sentimentanalyse

bert_cols = [
    "bert_sentiment_discrete",
    "bert_sentiment_cont",
    "bert_confidence",
    "bert_p1",
    "bert_p2",
    "bert_p3",
    "bert_p4",
    "bert_p5"
]

df[bert_cols] = df["review_text_clean"].progress_apply(get_bert_sentiment)

  0%|          | 0/1000 [00:00<?, ?it/s]

In [7]:
def get_bert_sentiment(text):
    if not isinstance(text, str) or text.strip() == "":
        return pd.Series([None, None, None, None, None, None, None])

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits.detach().numpy()[0]
    probs = softmax(logits)

    bert_discrete = int(probs.argmax() + 1)
    bert_continuous = sum((i + 1) * probs[i] for i in range(5))
    confidence = probs.max()

    return pd.Series([
        bert_discrete,
        bert_continuous,
        confidence,
        probs[0],
        probs[1],
        probs[2],
        probs[3],
        probs[4]
    ])

In [ ]:
### Export des erweiterten Datensatzes

df.to_excel("Bewertungen_mit_BERT.xlsx", index=False)

In [14]:
### Kontrolle des erweiterten Datensatzes

print(df.shape)

print(df["good_type"].value_counts())

print(df["stars"].value_counts().sort_index())

print(df["bert_sentiment_discrete"].value_counts().sort_index())

(1000, 16)
good_type
search        500
experience    500
Name: count, dtype: int64
stars
1     55
2     12
3     43
4     80
5    810
Name: count, dtype: int64
bert_sentiment_discrete
1.0     51
2.0     46
3.0     65
4.0    201
5.0    637
Name: count, dtype: int64


In [15]:
## Tabelle 11

(df["stars"] > df["bert_sentiment_discrete"]).mean()

np.float64(0.247)

In [22]:
(df["stars"] < df["bert_sentiment_discrete"]).mean()

np.float64(0.062)

In [23]:
(df["stars"] == df["bert_sentiment_discrete"]).mean()

np.float64(0.691)

In [24]:
search = df[df["good_type"]=="search"]
experience = df[df["good_type"]=="experience"]

search_match = (
    search["stars"] ==
    search["bert_sentiment_discrete"]
).mean()

experience_match = (
    experience["stars"] ==
    experience["bert_sentiment_discrete"]
).mean()

print(search_match)
print(experience_match)

0.67
0.712


In [26]:
pd.crosstab(
    search["stars"],
    search["bert_sentiment_discrete"]
)

bert_sentiment_discrete,1.0,2.0,3.0,4.0,5.0
stars,,,,,
1,24,9,1,0,1
2,1,0,2,0,0
3,2,8,7,4,0
4,1,4,10,15,14
5,4,4,16,84,289


In [27]:
pd.crosstab(
    experience["stars"],
    experience["bert_sentiment_discrete"]
)

bert_sentiment_discrete,1.0,2.0,3.0,4.0,5.0
stars,,,,,
1,10,7,1,0,2
2,2,2,5,0,0
3,3,5,9,4,1
4,1,1,7,16,11
5,3,6,7,78,319


In [3]:
## Grafische Darstellung

SEARCH_COLOR = "steelblue"

EXP_FACE = "white"
EXP_EDGE = "navy"
EXP_HATCH = "////"

In [16]:
pd.crosstab(
    df["stars"],
    df["bert_sentiment_discrete"]
)

bert_sentiment_discrete,1.0,2.0,3.0,4.0,5.0
stars,,,,,
1,34,16,2,0,3
2,3,2,7,0,0
3,5,13,16,8,1
4,2,5,17,31,25
5,7,10,23,162,608


In [70]:
pd.crosstab(
    df["stars"],
    df["bert_sentiment_discrete"],
    normalize="index"
).round(3) * 100

bert_sentiment_discrete,1.0,2.0,3.0,4.0,5.0
stars,,,,,
1,61.8,29.1,3.6,0.0,5.5
2,25.0,16.7,58.3,0.0,0.0
3,11.6,30.2,37.2,18.6,2.3
4,2.5,6.2,21.2,38.8,31.2
5,0.9,1.2,2.8,20.0,75.1


In [72]:
pd.crosstab(
    df[df["good_type"]=="search"]["stars"],
    df[df["good_type"]=="search"]["bert_sentiment_discrete"]
)


bert_sentiment_discrete,1.0,2.0,3.0,4.0,5.0
stars,,,,,
1,24,9,1,0,1
2,1,0,2,0,0
3,2,8,7,4,0
4,1,4,10,15,14
5,4,4,16,84,289


In [73]:
pd.crosstab(
    df[df["good_type"]=="experience"]["stars"],
    df[df["good_type"]=="experience"]["bert_sentiment_discrete"]
)

bert_sentiment_discrete,1.0,2.0,3.0,4.0,5.0
stars,,,,,
1,10,7,1,0,2
2,2,2,5,0,0
3,3,5,9,4,1
4,1,1,7,16,11
5,3,6,7,78,319


In [74]:
search_table = pd.crosstab(
    df[df["good_type"]=="search"]["stars"],
    df[df["good_type"]=="search"]["bert_sentiment_discrete"],
    normalize="index"
) * 100

In [17]:
### Tabelle 13

for good in ["search", "experience"]:

    print("\n", good.upper())

    for star in range(1, 6):

        subset = df[
            (df["good_type"] == good)
            & (df["stars"] == star)
        ]

        agreement = (
            (subset["stars"] == subset["bert_sentiment_discrete"])
            .mean()
            * 100
        )

        print(f"{star} Sterne: {agreement:.1f}%")


 SEARCH
1 Sterne: 68.6%
2 Sterne: 0.0%
3 Sterne: 33.3%
4 Sterne: 34.1%
5 Sterne: 72.8%

 EXPERIENCE
1 Sterne: 50.0%
2 Sterne: 22.2%
3 Sterne: 40.9%
4 Sterne: 44.4%
5 Sterne: 77.2%
